In [1]:
# ============== 导入必要库 ==============
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Jupyter 环境显示配置
from IPython.display import display, Markdown
import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, HBox, VBox, Output

# 启用 Plotly 在 Jupyter 中的交互式显示
import plotly.io as pio
pio.renderers.default = "notebook_connected"  # 或 "notebook", "colab"

In [2]:
# ============== 核心分析函数（与 Streamlit 版本保持一致） ==============

def load_stock_data(ticker, start_date, end_date, benchmark="None"):
    """加载股票数据和基准数据"""
    try:
        stock = yf.Ticker(ticker)
        data = stock.history(start=start_date, end=end_date)
        info = stock.info
        
        if benchmark != "None":
            benchmark_data = yf.download(benchmark, start=start_date, end=end_date, progress=False)
            if not benchmark_data.empty and 'Close' in benchmark_data.columns:
                data[f'{benchmark}_Return'] = benchmark_data['Close'].pct_change()
        
        return data, info
    except Exception as e:
        print(f"❌ 数据加载失败: {str(e)}")
        return pd.DataFrame(), {}


def calculate_metrics(data):
    """计算关键指标"""
    close = data["Close"]
    returns = close.pct_change()
    
    total_return = (close.iloc[-1] / close.iloc[0] - 1) * 100
    annual_vol = returns.std() * np.sqrt(252) * 100
    sharpe_ratio = (returns.mean() * 252) / (returns.std() * np.sqrt(252)) if returns.std() > 0 else 0
    max_drawdown = (close / close.expanding().max() - 1).min() * 100
    
    return {
        'total_return': total_return,
        'annual_vol': annual_vol,
        'sharpe_ratio': sharpe_ratio,
        'max_drawdown': max_drawdown,
        'returns': returns
    }


def add_moving_averages(data, periods):
    """添加移动平均线"""
    for period in periods:
        data[f'MA{period}'] = data['Close'].rolling(window=period).mean()
    return data


def calculate_rsi(close, period=14):
    """计算 RSI 指标"""
    delta = close.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi


def calculate_macd(close):
    """计算 MACD 指标"""
    exp1 = close.ewm(span=12, adjust=False).mean()
    exp2 = close.ewm(span=26, adjust=False).mean()
    macd = exp1 - exp2
    signal = macd.ewm(span=9, adjust=False).mean()
    histogram = macd - signal
    return macd, signal, histogram

In [3]:
# ============== 可视化函数 ==============

def plot_price_chart(data, ticker, ma_periods, show_volume=True):
    """绘制价格图表"""
    fig = go.Figure()
    
    fig.add_trace(go.Scatter(
        x=data.index,
        y=data["Close"],
        mode='lines',
        name='Close Price',
        line=dict(color='#3B82F6', width=2)
    ))
    
    for period in ma_periods:
        if f'MA{period}' in data.columns:
            fig.add_trace(go.Scatter(
                x=data.index,
                y=data[f'MA{period}'],
                mode='lines',
                name=f'{period}-Day MA',
                line=dict(width=1, dash='dash')
            ))
    
    if show_volume and 'Volume' in data.columns:
        fig.add_trace(go.Bar(
            x=data.index,
            y=data['Volume'],
            name='Volume',
            yaxis='y2',
            opacity=0.3,
            marker_color='#94A3B8'
        ))
        fig.update_layout(yaxis2=dict(title="Volume", overlaying="y", side="right", showgrid=False))
    
    fig.update_layout(
        title=f"{ticker} Price Chart",
        xaxis_title="Date",
        yaxis_title="Price (USD)",
        hovermode="x unified",
        height=500,
        template="plotly_white"
    )
    return fig


def plot_returns_distribution(returns):
    """绘制收益率分布直方图"""
    fig = go.Figure()
    fig.add_trace(go.Histogram(
        x=returns.dropna() * 100,
        nbinsx=50,
        name='Daily Returns',
        marker_color='#3B82F6',
        opacity=0.7
    ))
    fig.update_layout(
        title="Daily Returns Distribution",
        xaxis_title="Daily Return (%)",
        yaxis_title="Frequency",
        height=400,
        template="plotly_white"
    )
    return fig


def plot_technical_indicators(data, ticker, show_rsi=True, show_macd=True):
    """绘制技术指标子图"""
    rows = 1 + sum([show_rsi, show_macd])
    fig = make_subplots(
        rows=rows, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.1,
        subplot_titles=["Price Chart"] + 
        (["RSI"] if show_rsi else []) + 
        (["MACD"] if show_macd else [])
    )
    
    row_idx = 1
    # OHLC 蜡烛图
    fig.add_trace(
        go.Candlestick(
            x=data.index,
            open=data['Open'],
            high=data['High'],
            low=data['Low'],
            close=data['Close'],
            name="OHLC"
        ),
        row=row_idx, col=1
    )
    row_idx += 1
    
    # RSI
    if show_rsi:
        rsi = calculate_rsi(data['Close'])
        fig.add_trace(
            go.Scatter(x=data.index, y=rsi, name="RSI", line=dict(color='#8B5CF6', width=2)),
            row=row_idx, col=1
        )
        fig.add_hline(y=70, line_dash="dash", line_color="red", opacity=0.5, row=row_idx, col=1)
        fig.add_hline(y=30, line_dash="dash", line_color="green", opacity=0.5, row=row_idx, col=1)
        row_idx += 1
    
    # MACD
    if show_macd:
        macd, signal, histogram = calculate_macd(data['Close'])
        fig.add_trace(go.Scatter(x=data.index, y=macd, name="MACD", line=dict(color='#3B82F6', width=2)), row=row_idx, col=1)
        fig.add_trace(go.Scatter(x=data.index, y=signal, name="Signal", line=dict(color='#EF4444', width=2)), row=row_idx, col=1)
        fig.add_trace(
            go.Bar(x=data.index, y=histogram, name="Histogram", 
                  marker_color=np.where(histogram > 0, '#10B981', '#EF4444')),
            row=row_idx, col=1
        )
    
    fig.update_layout(
        height=200 * rows,
        showlegend=True,
        template="plotly_white",
        xaxis_rangeslider_visible=False
    )
    return fig


def plot_benchmark_comparison(data, ticker, benchmark, returns):
    """绘制基准对比图"""
    if f'{benchmark}_Return' not in data.columns:
        print(f"⚠️ 未找到 {benchmark} 数据")
        return None
    
    cum_stock_return = (1 + returns).cumprod() - 1
    cum_bench_return = (1 + data[f'{benchmark}_Return']).cumprod() - 1
    
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=cum_stock_return.index,
        y=cum_stock_return * 100,
        mode='lines',
        name=ticker,
        line=dict(color='#3B82F6', width=2)
    ))
    fig.add_trace(go.Scatter(
        x=cum_bench_return.index,
        y=cum_bench_return * 100,
        mode='lines',
        name=benchmark,
        line=dict(color='#EF4444', width=2)
    ))
    
    fig.update_layout(
        title=f"Cumulative Returns: {ticker} vs {benchmark}",
        xaxis_title="Date",
        yaxis_title="Cumulative Return (%)",
        height=400,
        template="plotly_white"
    )
    return fig

In [4]:
# ============== Jupyter 交互式控件 ==============

def create_interactive_analysis():
    """创建交互式分析界面"""
    
    # 输入控件
    ticker_widget = widgets.Text(value='AAPL', description='Ticker:', disabled=False)
    start_date_widget = widgets.DatePicker(description='Start Date', value=datetime.now() - timedelta(days=365))
    end_date_widget = widgets.DatePicker(description='End Date', value=datetime.now())
    
    # 选项控件
    show_advanced_widget = widgets.Checkbox(value=True, description='Show Advanced Metrics')
    show_volume_widget = widgets.Checkbox(value=True, description='Show Volume')
    ma_periods_widget = widgets.SelectMultiple(
        options=[5, 20, 50, 100, 200],
        value=[20, 50],
        description='MA Periods:',
        disabled=False
    )
    show_rsi_widget = widgets.Checkbox(value=True, description='Show RSI')
    show_macd_widget = widgets.Checkbox(value=True, description='Show MACD')
    benchmark_widget = widgets.Dropdown(options=["SPY", "QQQ", "DIA", "IWM", "None"], value="None", description='Benchmark:')
    
    # 输出区域
    output = Output()
    
    def run_analysis(b):
        """执行分析的主函数"""
        with output:
            output.clear_output()
            
            ticker = ticker_widget.value.upper()
            start_date = start_date_widget.value
            end_date = end_date_widget.value
            ma_periods = list(ma_periods_widget.value)
            benchmark = benchmark_widget.value
            
            print(f"🔍 分析 {ticker} | 时间: {start_date} 至 {end_date}")
            print("-" * 60)
            
            # 加载数据
            data, info = load_stock_data(ticker, start_date, end_date, benchmark)
            if data.empty:
                print("❌ 未获取到数据，请检查代码或日期范围")
                return
            
            # 添加移动平均线
            data = add_moving_averages(data, ma_periods)
            metrics = calculate_metrics(data)
            
            # 📊 显示基本信息
            display(Markdown(f"### 📈 {ticker} - {info.get('longName', 'N/A')}"))
            display(Markdown(f"**行业**: {info.get('industry', 'N/A')} | **板块**: {info.get('sector', 'N/A')}"))
            display(Markdown(f"**市值**: ${info.get('marketCap', 0):,.0f} | **Forward P/E**: {info.get('forwardPE', 'N/A')}"))
            
            # 📋 显示关键指标
            print("\n📊 Key Metrics:")
            print(f"• Total Return: {metrics['total_return']:+.2f}%")
            print(f"• Annual Volatility: {metrics['annual_vol']:.2f}%")
            if show_advanced_widget.value:
                print(f"• Sharpe Ratio: {metrics['sharpe_ratio']:.2f}")
                print(f"• Max Drawdown: {metrics['max_drawdown']:.2f}%")
            
            # 📈 价格图表
            display(Markdown("\n### 📈 Price Chart"))
            fig_price = plot_price_chart(data, ticker, ma_periods, show_volume_widget.value)
            display(fig_price)
            
            # 📊 收益率分布
            display(Markdown("\n### 📊 Daily Returns Distribution"))
            fig_hist = plot_returns_distribution(metrics['returns'])
            display(fig_hist)
            
            # 📋 统计信息
            returns = metrics['returns']
            print("\n📋 Return Statistics:")
            stats = {
                "Mean (%)": f"{returns.mean() * 100:.4f}",
                "Std Dev (%)": f"{returns.std() * 100:.4f}",
                "Skewness": f"{returns.skew():.4f}",
                "Kurtosis": f"{returns.kurtosis():.4f}",
                "Min (%)": f"{returns.min() * 100:.4f}",
                "Max (%)": f"{returns.max() * 100:.4f}",
                "Positive Days": f"{(returns > 0).sum()}",
                "Negative Days": f"{(returns < 0).sum()}"
            }
            for k, v in stats.items():
                print(f"• {k}: {v}")
            
            # 🔧 技术指标
            if show_rsi_widget.value or show_macd_widget.value:
                display(Markdown("\n### 🔧 Technical Indicators"))
                fig_tech = plot_technical_indicators(data, ticker, show_rsi_widget.value, show_macd_widget.value)
                display(fig_tech)
            
            # 🆚 基准对比
            if benchmark != "None":
                display(Markdown(f"\n### 🆚 vs {benchmark} Comparison"))
                fig_comp = plot_benchmark_comparison(data, ticker, benchmark, metrics['returns'])
                if fig_comp:
                    display(fig_comp)
            
            # 📄 原始数据预览
            display(Markdown("\n### 📄 Raw Data Preview (last 10 rows)"))
            display_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
            display(data[display_cols].tail(10).style.format({
                'Open': '{:.2f}', 'High': '{:.2f}', 'Low': '{:.2f}', 
                'Close': '{:.2f}', 'Volume': '{:,.0f}'
            }))
            
            # 💾 导出选项
            csv = data.to_csv().encode('utf-8')
            from IPython.display import FileLink
            import os
            filename = f"{ticker}_stock_data.csv"
            with open(filename, 'wb') as f:
                f.write(csv)
            print(f"\n✅ 数据已保存为 {filename}")
            display(FileLink(filename))
    
    # 运行按钮
    run_button = widgets.Button(description="🚀 Run Analysis", button_style='primary')
    run_button.on_click(run_analysis)
    
    # 界面布局
    controls = VBox([
        HBox([ticker_widget, benchmark_widget]),
        HBox([start_date_widget, end_date_widget]),
        HBox([show_advanced_widget, show_volume_widget]),
        ma_periods_widget,
        HBox([show_rsi_widget, show_macd_widget]),
        run_button
    ])
    
    display(VBox([controls, output]))

In [5]:
# ============== 可视化函数 ==============

def plot_price_chart(data, ticker, ma_periods, show_volume=True):
    """绘制价格图表"""
    fig = go.Figure()
    
    fig.add_trace(go.Scatter(
        x=data.index,
        y=data["Close"],
        mode='lines',
        name='Close Price',
        line=dict(color='#3B82F6', width=2)
    ))
    
    for period in ma_periods:
        if f'MA{period}' in data.columns:
            fig.add_trace(go.Scatter(
                x=data.index,
                y=data[f'MA{period}'],
                mode='lines',
                name=f'{period}-Day MA',
                line=dict(width=1, dash='dash')
            ))
    
    if show_volume and 'Volume' in data.columns:
        fig.add_trace(go.Bar(
            x=data.index,
            y=data['Volume'],
            name='Volume',
            yaxis='y2',
            opacity=0.3,
            marker_color='#94A3B8'
        ))
        fig.update_layout(yaxis2=dict(title="Volume", overlaying="y", side="right", showgrid=False))
    
    fig.update_layout(
        title=f"{ticker} Price Chart",
        xaxis_title="Date",
        yaxis_title="Price (USD)",
        hovermode="x unified",
        height=500,
        template="plotly_white"
    )
    return fig


def plot_returns_distribution(returns):
    """绘制收益率分布直方图"""
    fig = go.Figure()
    fig.add_trace(go.Histogram(
        x=returns.dropna() * 100,
        nbinsx=50,
        name='Daily Returns',
        marker_color='#3B82F6',
        opacity=0.7
    ))
    fig.update_layout(
        title="Daily Returns Distribution",
        xaxis_title="Daily Return (%)",
        yaxis_title="Frequency",
        height=400,
        template="plotly_white"
    )
    return fig


def plot_technical_indicators(data, ticker, show_rsi=True, show_macd=True):
    """绘制技术指标子图"""
    rows = 1 + sum([show_rsi, show_macd])
    fig = make_subplots(
        rows=rows, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.1,
        subplot_titles=["Price Chart"] + 
        (["RSI"] if show_rsi else []) + 
        (["MACD"] if show_macd else [])
    )
    
    row_idx = 1
    # OHLC 蜡烛图
    fig.add_trace(
        go.Candlestick(
            x=data.index,
            open=data['Open'],
            high=data['High'],
            low=data['Low'],
            close=data['Close'],
            name="OHLC"
        ),
        row=row_idx, col=1
    )
    row_idx += 1
    
    # RSI
    if show_rsi:
        rsi = calculate_rsi(data['Close'])
        fig.add_trace(
            go.Scatter(x=data.index, y=rsi, name="RSI", line=dict(color='#8B5CF6', width=2)),
            row=row_idx, col=1
        )
        fig.add_hline(y=70, line_dash="dash", line_color="red", opacity=0.5, row=row_idx, col=1)
        fig.add_hline(y=30, line_dash="dash", line_color="green", opacity=0.5, row=row_idx, col=1)
        row_idx += 1
    
    # MACD
    if show_macd:
        macd, signal, histogram = calculate_macd(data['Close'])
        fig.add_trace(go.Scatter(x=data.index, y=macd, name="MACD", line=dict(color='#3B82F6', width=2)), row=row_idx, col=1)
        fig.add_trace(go.Scatter(x=data.index, y=signal, name="Signal", line=dict(color='#EF4444', width=2)), row=row_idx, col=1)
        fig.add_trace(
            go.Bar(x=data.index, y=histogram, name="Histogram", 
                  marker_color=np.where(histogram > 0, '#10B981', '#EF4444')),
            row=row_idx, col=1
        )
    
    fig.update_layout(
        height=200 * rows,
        showlegend=True,
        template="plotly_white",
        xaxis_rangeslider_visible=False
    )
    return fig


def plot_benchmark_comparison(data, ticker, benchmark, returns):
    """绘制基准对比图"""
    if f'{benchmark}_Return' not in data.columns:
        print(f"⚠️ 未找到 {benchmark} 数据")
        return None
    
    cum_stock_return = (1 + returns).cumprod() - 1
    cum_bench_return = (1 + data[f'{benchmark}_Return']).cumprod() - 1
    
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=cum_stock_return.index,
        y=cum_stock_return * 100,
        mode='lines',
        name=ticker,
        line=dict(color='#3B82F6', width=2)
    ))
    fig.add_trace(go.Scatter(
        x=cum_bench_return.index,
        y=cum_bench_return * 100,
        mode='lines',
        name=benchmark,
        line=dict(color='#EF4444', width=2)
    ))
    
    fig.update_layout(
        title=f"Cumulative Returns: {ticker} vs {benchmark}",
        xaxis_title="Date",
        yaxis_title="Cumulative Return (%)",
        height=400,
        template="plotly_white"
    )
    return fig

In [6]:
# ============== 运行分析 ==============
# 执行此行即可启动交互式分析界面
create_interactive_analysis()